# Lesson 17 Lab — Speculative Decoding and Acceptance

**Puzzle:** When does proposing several tokens reduce ITL instead of adding verification overhead?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Speculative decoding accelerates memory-bound Decode only when cheap proposed tokens are accepted often enough. A method name or proposal length cannot predict the result without acceptance and target-model timing.


## 0. Predict before running

1. Predict acceptance on a repeated sequence prompt.
2. Check greedy token equality between configurations.
3. State why one elapsed-time sample is insufficient for promotion.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The native lab compares ordinary decoding with prompt-lookup n-gram speculation on a repetitive prompt, using identical greedy sampling. It records success, elapsed time, token equality, and exposed acceptance metrics.

- Speculation changes execution, not the target distribution contract.
- Acceptance rate is workload-dependent.
- High-QPS batching can reduce the relative value of speculative Decode.


## 2. Derive the mechanism

A proposer emits multiple candidate tokens. The target verifies them in a batched pass and accepts the valid prefix; rejected positions resume target decoding. N-gram lookup proposes repeated prompt continuations without a draft model. Expected benefit depends on proposal cost, verification efficiency, acceptance length, and offered load.

### Mechanism at a glance

```mermaid
flowchart LR
  C["current context"] --> P["cheap proposer: k tokens"]
  P --> V["target verifies candidates"]
  V --> A{"accepted prefix"}
  A -->|"many accepted"| F["advance several positions"]
  A -->|"early rejection"| R["resume target decode"]
  F --> C
  R --> C
```

### Walk it step by step

1. **Select a proposer.** Match draft, n-gram, suffix, or MTP to available artifacts.
2. **Verify with the target.** Acceptance preserves the target distribution contract.
3. **Measure accepted progress.** Count how many target positions advance per verification.
4. **Sweep offered load.** Compare ITL and throughput where Decode is actually the bottleneck.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 17
LESSON_TITLE = 'Speculative Decoding and Acceptance'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260829
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | ordinary target-model Decode |
| Candidate | n-gram prompt-lookup speculation with four proposed tokens |
| Held constant | model, prompt, greedy sampling, maximum tokens, engine limits, and GPU |
| Measurements | success, elapsed time, output tokens, token equality, and acceptance counters when exposed |
| Evidence | `native-backend` |

**Experiment:** Run matched native baseline and n-gram speculative engines, retaining token and timing evidence.


## 5. Inspect the experiment code

The two engines are created sequentially to avoid shared VRAM. The speculative config follows the installed release schema and any incompatibility is kept as a structured failure.

Do not execute until the code matches the frozen table.


In [2]:
prompt=("red blue green red blue green "*35)+"Continue:"; params=SamplingParams(temperature=0.0,max_tokens=32,seed=SEED)
def run_engine(extra):
    row={"success":False,"elapsed_s":None,"output_tokens":0,"token_ids":[],"error":None}
    try:
        engine=LLM(**base_engine_args(max_model_len=1024,**extra)); tick=time.perf_counter()
        item=engine.generate([prompt],params,use_tqdm=False)[0]; record=output_record(item)
        row.update(success=True,elapsed_s=time.perf_counter()-tick,output_tokens=record["output_tokens"],
                   token_ids=record["token_ids"],num_cached_tokens=record["num_cached_tokens"])
        del engine; gc.collect(); torch.cuda.empty_cache()
    except Exception as exc: row["error"]=f"{type(exc).__name__}: {exc}"; gc.collect(); torch.cuda.empty_cache()
    return row
baseline=run_engine({}); speculative=run_engine({"speculative_config":{"method":"ngram",
    "num_speculative_tokens":4,"prompt_lookup_min":2,"prompt_lookup_max":5}})
ratio=baseline["elapsed_s"]/speculative["elapsed_s"] if baseline["success"] and speculative["success"] else None
metrics={"baseline":baseline,"speculative":speculative,
         "tokens_equal":bool(baseline["success"] and speculative["success"] and baseline["token_ids"]==speculative["token_ids"]),
         "speed_ratio":ratio}
analysis=(f"Baseline/speculative success={baseline['success']}/{speculative['success']}, tokens equal="
          f"{metrics['tokens_equal']}, elapsed ratio={ratio}. The repeated prompt is favorable to n-gram lookup.")


INFO 08-13 00:21:41 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260829, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:21:41 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:21:41 [model.py:1883] Using max model len 1024


INFO 08-13 00:21:41 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:21:41 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=652361) INFO 08-13 00:22:11 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.92 s


(EngineCore pid=652361) WARNING 08-13 00:22:12 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=652361) WARNING 08-13 00:22:12 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=652361) INFO 08-13 00:22:12 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=652361) INFO 08-13 00:22:12 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=652361) INFO 08-13 00:22:12 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


(EngineCore pid=652361) <remote-home>/vllm-ch03/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
(EngineCore pid=652361)   warnings.warn(problem)


(EngineCore pid=652361) WARNING 08-13 00:22:13 [jit_monitor.py:135] Triton kernel JIT compilation during inference: rejection_greedy_sample_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
INFO 08-13 00:22:13 [utils.py:612] [shutdown] Process manager: send sigterm to process EngineCore


(EngineCore pid=652361) INFO 08-13 00:22:13 [core.py:1332] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=652361) INFO 08-13 00:22:13 [core.py:1468] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=652361) INFO 08-13 00:22:13 [core.py:1499] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=652361) INFO 08-13 00:22:13 [core.py:1345] [shutdown] EngineCore: exiting busy loop


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Baseline success | yes |
| Speculative success | yes |
| Tokens equal | yes |
| Baseline elapsed | 0.347510 |
| Speculative elapsed | 1.409890 |
| Speed ratio | 0.246x |
| Output tokens | 32 |


## 7. Explain the result

Baseline/speculative success=True/True, tokens equal=True, elapsed ratio=0.24648057002129478. The repeated prompt is favorable to n-gram lookup.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 17, "title": 'Speculative Decoding and Acceptance', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Speculation is valuable only when accepted target work offsets proposer and verification cost; this native pair bounds the claim to one repetitive workload.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 17,
  "title": "Speculative Decoding and Acceptance",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260829
  },
  "evidence_label": "native-backend",
  "metrics": {
    "baseline": {
      "success": true,
      "elapsed_s": 0.3475104998797178,
      "output_tokens": 32,
      "token_ids": [
        220,
        16,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15,
        15
      ],
      "error": null,
      "num_cached_tokens": 0
    },
    "speculative": {
      "success": 

## 9. Make the bounded decision

> Speculation is valuable only when accepted target work offsets proposer and verification cost; this native pair bounds the claim to one repetitive workload.

**Acceptance/rollback:** Enable speculation only when representative low/medium-QPS traffic improves ITL without quality, throughput, or memory regressions.

**Failure analysis:** A repetitive prompt favors n-gram lookup and is not representative. Compilation warm-up, batching, and version-specific metrics can dominate a small run.


## 10. Extend the evidence

Benchmark several prompt families and arrival rates, collect proposer/acceptance counters, and compare p50/p95 ITL after warm-up.

The full boundary and references are in [`README.md`](README.md).
